In [3]:
import pandas as pd
from pathlib import Path

In [6]:
DATASET_DIR = Path("../Demos/medical_datasets/whole_multicare_dataset")

cases_path = DATASET_DIR / "cases.parquet"
metadata_path = DATASET_DIR / "metadata.parquet"

print(f"[*] Cases' path: {cases_path.resolve()}")
print(f"[*] Does the path exist? {cases_path.exists()}")

[*] Cases' path: /home/aminh/Projects/LocalAIProject/MultiCaRe_Dataset/Demos/medical_datasets/whole_multicare_dataset/cases.parquet
[*] Does the path exist? True


In [14]:
df_sample = pd.read_parquet(cases_path).head(10)
print("Columns in the dataset:", df_sample.columns.tolist())

first_case = df_sample['cases'].iloc[0][0]
for k in first_case.keys():
    print(f" - {k}")

Columns in the dataset: ['cases', 'article_id']
 - age
 - case_id
 - case_text
 - gender


In [15]:
print("[*] Loading all cases.parquet into memory...")
df_cases = pd.read_parquet(cases_path)

print(f"[+] Successfully loaded: {len(df_cases):,} cases (articles)")

[*] Loading all cases.parquet into memory...
[+] Successfully loaded: 76,137 cases (articles)


In [5]:
"""
DISEASE_RULES: Bộ từ điển quy tắc gán nhãn Tầng 1 (Etiology-Grounded Filter)
Nguồn tham chiếu y khoa:
- CDC DPDx (Laboratory Identification of Parasites of Public Health Concern)
- CDC NNDSS (National Notifiable Diseases Surveillance System Case Definitions)
- WHO Technical Report Series & Disease Specific Diagnostic Guidelines
"""

DISEASE_RULES = {
    # 1. Bệnh Lao (Tuberculosis) - Ref: WHO Module 3 Diagnosis & CDC NNDSS
    "Tuberculosis": {
        "source": "WHO Consolidated Guidelines on Tuberculosis / CDC NNDSS",
        "pathogens": [
            "mycobacterium tuberculosis", "m. tuberculosis", "mtb complex", 
            "mycobacterium bovis"
        ],
        "gold_standards": [
            "acid-fast bacilli", "afb smear", "afb stain", "ziehl-neelsen", 
            "genexpert", "xpert mtb", "lowenstein-jensen", "mycobacterial culture", 
            "mgit", "caseating granuloma"
        ]
    },

    # 2. Bệnh Melioidosis - Ref: CDC NNDSS Code 10227 & Dance DAB Lancet Inf Dis
    "Melioidosis": {
        "source": "CDC NNDSS Melioidosis Case Definition / The Lancet Inf Dis",
        "pathogens": [
            "burkholderia pseudomallei", "b. pseudomallei", "pseudomonas pseudomallei"
        ],
        "gold_standards": [
            "ashdown", "blood culture yielded", "pus culture yielded", 
            "vitek 2", "safety-pin appearance", "bipolar staining", 
            "isolation of burkholderia", "isolated from blood", "isolated from sputum"
        ]
    },

    # 3. Bệnh Amip (Amoebiasis) - Ref: CDC DPDx Amebiasis
    "Amoebiasis": {
        "source": "CDC DPDx - Amebiasis",
        "pathogens": [
            "entamoeba histolytica", "e. histolytica"
        ],
        "gold_standards": [
            "trophozoites with ingested erythrocytes", "erythrophagocytosis", 
            "anchovy paste", "e. histolytica pcr", "stool antigen detection", 
            "amebic serology", "colonic biopsy showing trophozoites"
        ]
    },

    # 4. Ấu trùng sán lợn thần kinh (Neurocysticercosis) - Ref: CDC DPDx & Del Brutto Criteria
    "Neurocysticercosis": {
        "source": "CDC DPDx - Cysticercosis / Del Brutto Consensus Criteria",
        "pathogens": [
            "taenia solium", "t. solium", "cysticercus cellulosae", "cysticercus"
        ],
        "gold_standards": [
            "scolex", "cystic lesion with scolex", "del brutto", 
            "western blot", "eitm", "brain biopsy showing cysticercus", 
            "histopathology demonstrated cyst wall", "calcified granuloma with scolex"
        ]
    },

    # 5. Nang sán gai (Echinococcosis / Hydatid Disease) - Ref: CDC DPDx & WHO IWGE
    "Echinococcosis": {
        "source": "CDC DPDx - Echinococcosis / WHO-IWGE",
        "pathogens": [
            "echinococcus granulosus", "echinococcus multilocularis", 
            "e. granulosus", "e. multilocularis", "hydatid disease"
        ],
        "gold_standards": [
            "protoscolices", "hooklets", "hydatid sand", "laminated layer", 
            "chitinous layer", "cystectomy showing scolices", "hydatid fluid examination",
            "pair procedure"
        ]
    },

    # 6. Sán máng (Schistosomiasis) - Ref: CDC DPDx Schistosomiasis
    "Schistosomiasis": {
        "source": "CDC DPDx - Schistosomiasis",
        "pathogens": [
            "schistosoma mansoni", "schistosoma haematobium", "schistosoma japonicum", 
            "s. mansoni", "s. haematobium", "s. japonicum"
        ],
        "gold_standards": [
            "lateral spine", "terminal spine", "schistosoma eggs", 
            "schistosome ova", "kato-katz", "rectal snip", "bladder biopsy showing eggs", 
            "urine filtration showing eggs"
        ]
    },

    # 7. Bệnh Phong / Hansen (Leprosy) - Ref: WHO Leprosy Guidelines 2018
    "Leprosy": {
        "source": "WHO Guidelines for diagnosis, treatment of leprosy 2018",
        "pathogens": [
            "mycobacterium leprae", "m. leprae", "mycobacterium lepromatosis"
        ],
        "gold_standards": [
            "slit-skin smear", "slit skin smear", "fite stain", "fite-faraco", 
            "acid-fast bacilli in nerves", "globi", "bacillary index", 
            "skin biopsy showing nerve involvement", "ridley-jopling"
        ]
    },

    # 8. Bệnh Leishmania (Leishmaniasis) - Ref: CDC DPDx & WHO TRS 949
    "Leishmaniasis": {
        "source": "CDC DPDx - Leishmaniasis / WHO Expert Committee",
        "pathogens": [
            "leishmania donovani", "leishmania infantum", "leishmania major", 
            "leishmania tropica", "leishmania braziliensis", "l. donovani", "l. infantum"
        ],
        "gold_standards": [
            "amastigote", "leishman-donovan bodies", "ld bodies", 
            "splenic aspirate", "bone marrow aspirate smear", "rk39", 
            "giemsa stain showing amastigotes", "nnn medium"
        ]
    },

    # 9. Nhiễm nấm Cryptococcus (Cryptococcosis) - Ref: WHO Cryptococcal Guidelines 2018
    "Cryptococcosis": {
        "source": "WHO Cryptococcal Guidelines 2018 / CDC Cryptococcosis",
        "pathogens": [
            "cryptococcus neoformans", "cryptococcus gattii", "c. neoformans", "c. gattii"
        ],
        "gold_standards": [
            "india ink", "mucicarmine", "cryptococcal antigen", "crag", 
            "encapsulated yeast", "bird seed agar", "csf fungal culture"
        ]
    },

    # 10. Nhiễm nấm Histoplasma (Histoplasmosis) - Ref: PAHO/WHO Guidelines 2020
    "Histoplasmosis": {
        "source": "PAHO/WHO Guidelines for Histoplasmosis 2020 / CDC",
        "pathogens": [
            "histoplasma capsulatum", "h. capsulatum"
        ],
        "gold_standards": [
            "intracellular yeast", "grocott methenamine silver", "gms stain", 
            "histoplasma antigen", "wright-giemsa showing yeast in macrophages", 
            "tuberculate macroconidia", "fungal culture yielding histoplasma"
        ]
    },

    # 11. Sán lá phổi (Paragonimiasis) - Ref: CDC DPDx Paragonimiasis
    "Paragonimiasis": {
        "source": "CDC DPDx - Paragonimiasis",
        "pathogens": [
            "paragonimus westermani", "paragonimus kellicotti", "p. westermani", "paragonimus"
        ],
        "gold_standards": [
            "operculated eggs", "operculated ova", "eggs in sputum", 
            "paragonimus eggs in stool", "bronchoalveolar lavage showing ova", 
            "lung biopsy showing fluke"
        ]
    },

    # 12. Giun chỉ bạch huyết (Lymphatic Filariasis) - Ref: CDC DPDx Filariasis
    "Lymphatic Filariasis": {
        "source": "CDC DPDx - Lymphatic Filariasis / WHO GPELF",
        "pathogens": [
            "wuchereria bancrofti", "brugia malayi", "brugia timori", 
            "w. bancrofti", "b. malayi"
        ],
        "gold_standards": [
            "microfilariae in nocturnal", "thick blood film", "sheathed microfilaria", 
            "filarial dance sign", "circulating filarial antigen", "cfa test", 
            "ict card test", "lymph node biopsy showing adult worm"
        ]
    },

    # 13. Sốt rét (Malaria) - Ref: CDC DPDx Malaria & WHO Malaria Guidelines
    "Malaria": {
        "source": "CDC DPDx - Malaria / WHO Guidelines for Malaria",
        "pathogens": [
            "plasmodium falciparum", "plasmodium vivax", "plasmodium malariae", 
            "plasmodium ovale", "plasmodium knowlesi", "p. falciparum", "p. vivax"
        ],
        "gold_standards": [
            "thick and thin blood film", "blood smear", "giemsa stain", 
            "ring forms", "trophozoite", "schizont", "gametocyte", 
            "rapid diagnostic test", "malaria pcr", "pfhrp2"
        ]
    },

    # 14. COVID-19 (SARS-CoV-2) - Ref: WHO COVID-19 Case Definitions & CDC
    "Covid19": {
        "source": "WHO COVID-19 Surveillance Case Definitions / CDC",
        "pathogens": [
            "sars-cov-2", "covid-19", "covid19", "2019-ncov", "novel coronavirus"
        ],
        "gold_standards": [
            "rt-pcr", "reverse transcription-polymerase chain reaction", 
            "naat", "nasopharyngeal swab pcr", "positive rna test", 
            "viral rna was detected"
        ]
    },

    # 15. Sốt xuất huyết Dengue - Ref: WHO Dengue Guidelines 2009 & CDC
    "Dengue": {
        "source": "WHO Dengue Guidelines for Diagnosis 2009 / CDC",
        "pathogens": [
            "dengue virus", "denv-1", "denv-2", "denv-3", "denv-4", "denv"
        ],
        "gold_standards": [
            "ns1 antigen", "dengue ns1", "dengue rt-pcr", "dengue rna", 
            "igm seroconversion", "four-fold rise in igg", "plaque reduction neutralization"
        ]
    }
}

print(f"[+] Successfully initiallized DISEASE_RULES for {len(DISEASE_RULES)} diseases!")


[+] Successfully initiallized DISEASE_RULES for 15 diseases!


In [20]:
filtered_cases = []

for article_id, cases_list in zip(df_cases['article_id'], df_cases['cases']):

    for case in cases_list:
        c_id = case['case_id']
        age = case.get('age')
        gender = case.get('gender')
        text = case.get('case_text', '')

        text_lower = text.lower()

        matched_for_this_case = []

        for disease_name, rule in DISEASE_RULES.items():

            has_pathogen = any(p in text_lower for p in rule["pathogens"])

            has_gold_standard = any(g in text_lower for g in rule["gold_standards"])

            if has_pathogen and has_gold_standard:
                matched_for_this_case.append(disease_name)

        if matched_for_this_case:
            filtered_cases.append({
                "article_id": article_id,
                "case_id": c_id,
                "age": age,
                "gender": gender,
                "matched_diseases": matched_for_this_case,
                "case_text": text
            })

print(f"[+] Successfully filtered. Found: {len(filtered_cases):,} useful cases")

[+] Successfully filtered. Found: 1,351 useful cases


In [ ]:
output_json_path = "tier1_filtered_cases.json"

df_tier1 = pd.DataFrame(filtered_cases)
df_tier1.to_parquet("tier1_filtered_cases.parquet", index=False)
print(f"[+] Successfully saved 1,351 cases to hard disk!")
print(f" - Parquet: tier1_filtered_cases.parquet")

[+] ĐÃ LƯU THÀNH CÔNG 1,351 CA XUỐNG Ổ CỨNG!
 - JSON: tier1_filtered_cases.json
 - Parquet: tier1_filtered_cases.parquet


In [2]:
import pandas as pd
df_tier1 = pd.read_parquet("tier1_filtered_cases.parquet")

In [7]:
counts = df_tier1['matched_diseases'].explode().value_counts()

print(f"{'Disease name':<25} | {'Cases found'}")
print("-" * 45)
for disease in DISEASE_RULES.keys():
    print(f"{disease:<25} | {counts.get(disease, 0):>5} cases")

Disease name              | Cases found
---------------------------------------------
Tuberculosis              |   406 cases
Melioidosis               |    11 cases
Amoebiasis                |     1 cases
Neurocysticercosis        |    12 cases
Echinococcosis            |    20 cases
Schistosomiasis           |    12 cases
Leprosy                   |    13 cases
Leishmaniasis             |    35 cases
Cryptococcosis            |    69 cases
Histoplasmosis            |    21 cases
Paragonimiasis            |     0 cases
Lymphatic Filariasis      |     5 cases
Malaria                   |    82 cases
Covid19                   |   672 cases
Dengue                    |    13 cases
